# 5. MODEL EVALUATION AND DIAGNOSTICS
## Daily Customer Churn Predictor · VivaMarket Brasil

---

**INPUT:** `../models/churn_model_YYYYMMDD.joblib`, `../data/processed/churn_features_YYYYMMDD.parquet`, and `../data/processed/churn_predictions_YYYYMMDD.parquet`

*The selected churn model, the full feature matrix, and the scored test snapshots produced in NB04.*

**OUTPUT:** `../data/processed/churn_diagnostics_YYYYMMDD.csv` and `../reports/model_diagnostics_YYYYMMDD.html`

*A business-facing diagnostic package covering ranking quality, calibration, temporal stability, and threshold trade-offs.*


---
## 5.1. STARTING SITUATION


NB04 selected the best-performing model under a temporal split on the canonical V2C formulation and produced a scored test population with provisional percentile-based risk tiers. Before moving into explainability and deployment, the project needs a more rigorous view of how reliable those scores really are under the still-positive-heavy v2 target.

This notebook therefore turns raw predictive output into a **decision-quality diagnostic layer**. The goal is to verify whether the model remains useful across future monthly snapshots, how concentrated risk is at the top of the ranking, and how calibration and threshold choices affect the retention workload.

---
## 5.2. NOTEBOOK OBJECTIVE


- **Business objective:** verify that the selected churn model prioritizes the right customers for retention actions and supports economically reasonable contact thresholds.
- **Analytical objective:** quantify ranking quality, calibration, temporal stability, and campaign concentration across the held-out test period for the canonical V2C line.

---
## 5.3. INITIAL SETUP

**What is done**

We load the libraries required for diagnostics, plotting, model loading, and HTML report generation.

**Why it is done**

NB05 must be reproducible and explicit because the evaluation stage is where modeling quality becomes a business decision.

**Expected result**

A stable environment with resolved paths, active logging, and report folders ready for diagnostic outputs.


In [1]:
import base64
import io
import logging
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve, roc_auc_score, roc_curve

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', force=True)
logger = logging.getLogger('nb05_model_evaluation')
logger.info('NB05 started: model evaluation and diagnostics.')

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


2026-05-06 07:41:51,413 | INFO | NB05 started: model evaluation and diagnostics.


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

run_date_tag = datetime.now(ZoneInfo('Europe/Paris')).strftime('%Y%m%d')
model_path = sorted(MODELS_DIR.glob('churn_model_*.joblib'))[-1]
feature_path = sorted(PROCESSED_DIR.glob('churn_features_*.parquet'))[-1]
prediction_path = sorted(PROCESSED_DIR.glob('churn_predictions_*.parquet'))[-1]
diagnostics_csv_path = PROCESSED_DIR / f'churn_diagnostics_{run_date_tag}.csv'
diagnostics_html_path = REPORTS_DIR / f'model_diagnostics_{run_date_tag}.html'

logger.info('Model path: %s', model_path)
logger.info('Feature path: %s', feature_path)
logger.info('Prediction path: %s', prediction_path)


2026-05-06 07:41:51,423 | INFO | Model path: /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/models/churn_model_20260506.joblib


2026-05-06 07:41:51,424 | INFO | Feature path: /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_features_20260506.parquet


2026-05-06 07:41:51,424 | INFO | Prediction path: /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_predictions_20260506.parquet


---
## 5.4. DATA RECONSTRUCTION FOR DIAGNOSTICS

**What is done**

We reload the model package, rebuild the same encoded feature space used in NB04, and align the test rows with the persisted scored output.

**Why it is done**

Deep diagnostics need access to both the full feature matrix and the production-like prediction file so that ranking, stability and threshold analysis remain fully auditable.

**Expected result**

A matched diagnostic table for the held-out test period, including probabilities, labels, snapshot dates and campaign tiers.


In [3]:
package = joblib.load(model_path)
feature_df = pd.read_parquet(feature_path)
feature_df['snapshot_date'] = pd.to_datetime(feature_df['snapshot_date'])
prediction_df = pd.read_parquet(prediction_path)
prediction_df['snapshot_date'] = pd.to_datetime(prediction_df['snapshot_date'])

test_keys = package['test_snapshot_keys']
feature_columns = package['feature_columns']
scored_model = package['model']
target_column = package.get('target_column', 'churn_v2_label' if 'churn_v2_label' in feature_df.columns else 'churn_90d_label')

leakage_columns = [
    'customer_unique_id', 'snapshot_key', 'snapshot_date', 'first_purchase_timestamp',
    'last_purchase_timestamp', 'future_orders_90d', 'future_revenue_90d', 'churn_90d_label',
    'churn_v2_label', 'next_purchase_timestamp', 'days_to_next_purchase', 'future_purchase_within_horizon'
]
test_df = feature_df[feature_df['snapshot_key'].astype(str).isin([str(k) for k in test_keys])].copy()
X_test = pd.get_dummies(
    test_df[[c for c in feature_df.columns if c not in leakage_columns]],
    columns=['customer_state'],
    dtype=float,
)
X_test = X_test.reindex(columns=feature_columns, fill_value=0.0)
test_df['recomputed_probability'] = scored_model.predict_proba(X_test)[:, 1]
test_df['observed_target'] = test_df[target_column].astype(int)

diagnostics_df = prediction_df.merge(
    test_df[[
        'customer_unique_id', 'snapshot_key', 'snapshot_date', 'recomputed_probability', 'observed_target'
    ]],
    on=['customer_unique_id', 'snapshot_key', 'snapshot_date'],
    how='left',
    validate='one_to_one',
)
diagnostics_df['observed_target'] = diagnostics_df['observed_target_x'].fillna(diagnostics_df['observed_target_y']).astype(int)
diagnostics_df = diagnostics_df.drop(columns=['observed_target_x', 'observed_target_y'])
diagnostics_df['probability_diff'] = diagnostics_df['churn_probability'] - diagnostics_df['recomputed_probability']
logger.info('Target column used for diagnostics: %s', target_column)
logger.info('Maximum scoring reconstruction difference: %.10f', diagnostics_df['probability_diff'].abs().max())
diagnostics_df.head()

2026-05-06 07:41:51,554 | INFO | Target column used for diagnostics: churn_v2_label


2026-05-06 07:41:51,555 | INFO | Maximum scoring reconstruction difference: 0.0000000000


,customer_unique_id,snapshot_key,snapshot_date,recency_days,total_orders,total_payment_value,orders_30d,orders_90d,churn_probability,risk_tier,selected_model,version_name,recomputed_probability,observed_target,probability_diff
0,004288347e5e88a27ded2bb23747066c,20180401,2018-04-01,77,2,354.3700,0.0000,1.0000,0.9101,LOW,xgboost,v2,0.9101,1,0.0000
1,00cc12a6d8b578b8ebd21ea4e2ae8b27,20180401,2018-04-01,376,2,126.2000,0.0000,0.0000,0.9545,LOW,xgboost,v2,0.9545,1,0.0000
2,011b4adcd54683b480c4d841250a987f,20180401,2018-04-01,45,2,236.3000,0.0000,1.0000,0.8569,LOW,xgboost,v2,0.8569,1,0.0000
3,013f4353d26bb05dc6652f1269458d8d,20180401,2018-04-01,124,2,356.3900,0.0000,0.0000,0.6468,LOW,xgboost,v2,0.6468,1,0.0000
4,015557c9912277312b9073947804a7ba,20180401,2018-04-01,335,2,315.1200,0.0000,0.0000,0.9673,MEDIUM,xgboost,v2,0.9673,1,0.0000


---
## 5.5. GLOBAL PERFORMANCE AND THRESHOLD TRADE-OFFS

**What is done**

We quantify overall ranking quality, threshold trade-offs, and campaign concentration at the top of the score distribution.

**Why it is done**

Retention budgets care about who appears first in the ranking, how many customers would be contacted, and what observed churn rate sits inside each operational slice.

**Expected result**

A concise metric package that links predictive performance to the High / Medium / Low retention framework.


In [4]:
def precision_at_top_fraction(y_true: pd.Series, scores: pd.Series, fraction: float) -> float:
    rank_df = pd.DataFrame({'y_true': y_true.to_numpy(), 'score': scores.to_numpy()})
    rank_df = rank_df.sort_values('score', ascending=False).reset_index(drop=True)
    cutoff = max(int(np.ceil(len(rank_df) * fraction)), 1)
    return float(rank_df.head(cutoff)['y_true'].mean())

y_true = diagnostics_df['observed_target'].astype(int)
y_score = diagnostics_df['churn_probability'].astype(float)

metric_rows = [
    ('roc_auc', roc_auc_score(y_true, y_score)),
    ('average_precision', average_precision_score(y_true, y_score)),
    ('brier_score', brier_score_loss(y_true, y_score)),
    ('label_prevalence', float(y_true.mean())),
    ('precision_at_top_1pct', precision_at_top_fraction(y_true, y_score, 0.01)),
    ('precision_at_top_5pct', precision_at_top_fraction(y_true, y_score, 0.05)),
    ('precision_at_top_10pct', precision_at_top_fraction(y_true, y_score, 0.10)),
    ('high_risk_share', float((diagnostics_df['risk_tier'] == 'HIGH').mean())),
    ('medium_risk_share', float((diagnostics_df['risk_tier'] == 'MEDIUM').mean())),
    ('low_risk_share', float((diagnostics_df['risk_tier'] == 'LOW').mean())),
]
summary_metrics = pd.DataFrame(metric_rows, columns=['metric', 'value'])
summary_metrics

,metric,value
0,roc_auc,0.8016
1,average_precision,0.9937
2,brier_score,0.0572
3,label_prevalence,0.9770
4,precision_at_top_1pct,1.0000
5,precision_at_top_5pct,1.0000
6,precision_at_top_10pct,0.9970
7,high_risk_share,0.2002
8,medium_risk_share,0.2998
9,low_risk_share,0.5000


In [5]:
quantile_grid = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
threshold_rows = []
for quantile in quantile_grid:
    threshold = float(diagnostics_df['churn_probability'].quantile(quantile))
    targeted = diagnostics_df['churn_probability'] >= threshold
    contacts = int(targeted.sum())
    threshold_rows.append({
        'score_quantile_cutoff': quantile,
        'score_threshold': threshold,
        'targeted_rows': contacts,
        'targeted_share': float(targeted.mean()),
        'observed_churn_rate': float(diagnostics_df.loc[targeted, 'observed_target'].mean()) if contacts else np.nan,
        'avg_total_payment_value': float(diagnostics_df.loc[targeted, 'total_payment_value'].mean()) if contacts else np.nan,
    })
threshold_df = pd.DataFrame(threshold_rows)
threshold_df

,score_quantile_cutoff,score_threshold,targeted_rows,targeted_share,observed_churn_rate,avg_total_payment_value
0,0.5000,0.9558,1673,0.5000,0.9958,261.9451
1,0.6000,0.9714,1339,0.4002,0.9963,245.4430
2,0.7000,0.9808,1004,0.3001,0.9980,243.7155
3,0.8000,0.9877,670,0.2002,0.9985,251.8046
4,0.9000,0.9929,335,0.1001,0.9970,248.0339
5,0.9500,0.9949,169,0.0505,1.0000,250.8551


---
## 5.6. TEMPORAL STABILITY AND CALIBRATION

**What is done**

We evaluate diagnostics by monthly snapshot and compare predicted scores with observed churn rates through calibration bins.

**Why it is done**

A model that looks good in aggregate can still drift month to month or overstate confidence in certain parts of the score distribution.

**Expected result**

A stable monthly view that reveals whether ranking quality and confidence remain usable for daily campaign operations.


In [6]:
monthly_backtest = (
    diagnostics_df.groupby('snapshot_key', observed=True)
    .apply(lambda frame: pd.Series({
        'rows_n': len(frame),
        'observed_churn_rate': frame['observed_target'].mean(),
        'avg_score': frame['churn_probability'].mean(),
        'precision_at_top_10pct': precision_at_top_fraction(frame['observed_target'].astype(int), frame['churn_probability'], 0.10),
        'average_precision': average_precision_score(frame['observed_target'].astype(int), frame['churn_probability']),
    }), include_groups=False)
    .reset_index()
)
monthly_backtest

,snapshot_key,rows_n,observed_churn_rate,avg_score,precision_at_top_10pct,average_precision
0,20180401,"1,551.0000",0.9761,0.8713,0.9936,0.9933
1,20180501,"1,795.0000",0.9777,0.8694,1.0000,0.9941


In [7]:
calibration_bins = pd.qcut(diagnostics_df['churn_probability'], q=10, duplicates='drop')
calibration_df = (
    diagnostics_df.assign(calibration_bin=calibration_bins)
    .groupby('calibration_bin', observed=True)
    .agg(
        customers=('customer_unique_id', 'nunique'),
        avg_predicted_probability=('churn_probability', 'mean'),
        observed_churn_rate=('observed_target', 'mean'),
    )
    .reset_index()
)
calibration_df['absolute_calibration_gap'] = (calibration_df['avg_predicted_probability'] - calibration_df['observed_churn_rate']).abs()
calibration_df

,calibration_bin,customers,avg_predicted_probability,observed_churn_rate,absolute_calibration_gap
0,"(0.0528, 0.591]",220,0.4027,0.8925,0.4898
1,"(0.591, 0.792]",246,0.7003,0.9731,0.2728
2,"(0.792, 0.878]",243,0.8403,0.9641,0.1238
3,"(0.878, 0.928]",266,0.9053,0.9821,0.0768
4,"(0.928, 0.956]",254,0.9440,0.9790,0.0351
5,"(0.956, 0.971]",245,0.9649,0.9940,0.0292
6,"(0.971, 0.981]",235,0.9761,0.9910,0.0149
7,"(0.981, 0.988]",226,0.9844,0.9970,0.0126
8,"(0.988, 0.993]",212,0.9903,1.0000,0.0097
9,"(0.993, 0.998]",185,0.9952,0.9970,0.0018


---
## 5.7. DIAGNOSTIC VISUALS AND HTML REPORT


In [8]:
def figure_to_base64(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', bbox_inches='tight', dpi=160)
    plt.close(fig)
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

roc_fpr, roc_tpr, _ = roc_curve(y_true, y_score)
pr_precision, pr_recall, _ = precision_recall_curve(y_true, y_score)
prob_true, prob_pred = calibration_curve(y_true, y_score, n_bins=10, strategy='quantile')

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].plot(roc_fpr, roc_tpr, label=f'ROC AUC = {roc_auc_score(y_true, y_score):.3f}', color='#1f77b4')
axes[0, 0].plot([0, 1], [0, 1], linestyle='--', color='grey')
axes[0, 0].set_title('ROC CURVE')
axes[0, 0].legend()

axes[0, 1].plot(pr_recall, pr_precision, color='#ff7f0e')
axes[0, 1].set_title('PRECISION-RECALL CURVE')
axes[0, 1].set_xlabel('Recall')
axes[0, 1].set_ylabel('Precision')

sns.lineplot(data=monthly_backtest, x='snapshot_key', y='precision_at_top_10pct', marker='o', ax=axes[1, 0])
axes[1, 0].set_title('MONTHLY PRECISION AT TOP 10%')
axes[1, 0].tick_params(axis='x', rotation=45)

axes[1, 1].plot(prob_pred, prob_true, marker='o', color='#2ca02c')
axes[1, 1].plot([0, 1], [0, 1], linestyle='--', color='grey')
axes[1, 1].set_title('CALIBRATION CURVE')
axes[1, 1].set_xlabel('Predicted probability')
axes[1, 1].set_ylabel('Observed churn rate')

plt.tight_layout()
diagnostics_chart = figure_to_base64(fig)

risk_summary = (
    diagnostics_df.groupby('risk_tier', observed=False)
    .agg(
        rows_n=('customer_unique_id', 'size'),
        customers_n=('customer_unique_id', 'nunique'),
        observed_churn_rate=('observed_target', 'mean'),
        avg_probability=('churn_probability', 'mean')
    )
    .reset_index()
)

html_parts = [
    '<html><head><meta charset="utf-8"><title>Model Diagnostics</title></head><body>',
    '<h1>MODEL DIAGNOSTICS REPORT</h1>',
    '<p><strong>Diagnostic context:</strong> Canonical V2C line with percentile-based provisional risk tiers.</p>',
    '<h2>Summary metrics</h2>', summary_metrics.to_html(index=False),
    '<h2>Risk-tier summary</h2>', risk_summary.to_html(index=False),
    '<h2>Quantile threshold trade-offs</h2>', threshold_df.to_html(index=False),
    '<h2>Monthly backtest</h2>', monthly_backtest.to_html(index=False),
    '<h2>Calibration bins</h2>', calibration_df.to_html(index=False),
    f'<h2>Diagnostic visuals</h2><img src="data:image/png;base64,{diagnostics_chart}" style="max-width:1100px;">',
    '</body></html>'
]
diagnostics_html_path.write_text('\n'.join(html_parts), encoding='utf-8')

diagnostics_export = pd.concat([
    summary_metrics.assign(section='summary'),
    threshold_df.assign(section='thresholds'),
    monthly_backtest.assign(section='monthly_backtest'),
    calibration_df.assign(section='calibration')
], ignore_index=True, sort=False)
diagnostics_export.to_csv(diagnostics_csv_path, index=False)
logger.info('Diagnostics CSV saved to %s', diagnostics_csv_path)
logger.info('Diagnostics HTML report saved to %s', diagnostics_html_path)

2026-05-06 07:41:51,667 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


2026-05-06 07:41:51,672 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


2026-05-06 07:41:52,138 | INFO | Diagnostics CSV saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_diagnostics_20260506.csv


2026-05-06 07:41:52,139 | INFO | Diagnostics HTML report saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/reports/model_diagnostics_20260506.html


---
## 5.8. NOTEBOOK CLOSURE


The diagnostic stage confirms whether the selected ranking is stable enough to drive retention actions. The main operational value of this notebook is that it reframes model quality as **campaign quality**: who gets contacted, how much churn concentrates in the top ranks, and how stable the score remains across future monthly snapshots.

Under the canonical V2C formulation, the diagnostics should still be interpreted with one explicit caveat: the target remains highly positive, so strong ranking metrics do not automatically mean that the operational base is perfectly resolved from a business-policy perspective.

The next notebook should explain *why* those customers score high risk by translating the model into SHAP-based churn drivers and segment-level narratives.